# Como-Article Ground/Vehicle MVP

Spec: `docs/specs/0006-como-article-ground-vehicle-mvp.md`

This is the canonical full-shard workflow for the current project. It uses a Spark-native prefilter before the Python extraction UDF, then writes the gold candidate dataset and compact analysis CSVs.

```text
Bronze corpus segments
  -> native Spark rlike prefilter for curated GROUND como article
  -> Python UDF extraction only on plausible rows
  -> gold Parquet and WebApp-ready CSV outputs
```


In [1]:
import os
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

REPO_ROOT


PosixPath('/home/jovyan/work')

In [2]:
import os
import tempfile
import zipfile

from pyspark.sql import SparkSession

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-como-article-ground-vehicle-optimized-mvp")
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(256 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))

spark.sparkContext.addPyFile(str(package_zip))
SPARK_MASTER, spark.version


('local[4]', '4.1.1')

In [3]:
from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, load_or_build_bronze_dataframe
from tal_qual.como_article_ground_vehicle import (
    COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH,
    COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    como_article_examples_dataframe,
    como_article_ground_counts_dataframe,
    como_article_ground_vehicle_counts_dataframe,
    como_article_review_sample_dataframe,
    como_article_vehicle_counts_dataframe,
    como_article_vehicle_ground_counts_dataframe,
    prefilter_como_article_ground_vehicle_bronze_dataframe,
    prepare_como_article_ground_vehicle_dataframe,
    write_como_article_csv_outputs,
    write_como_article_ground_vehicle_parquet,
)


## Load Or Build Bronze

The extractor expects the bronze schema from notebook 02. If bronze output is absent, this notebook builds it from the configured raw shard.


In [4]:
bronze_path = REPO_ROOT / os.environ.get("TAL_QUAL_BRONZE_PATH", str(BRONZE_OUTPUT_PATH))
raw_path = REPO_ROOT / os.environ.get("TAL_QUAL_RAW_CORPUS_INPUT", str(RAW_CORPUS_INPUT))

bronze_df = load_or_build_bronze_dataframe(spark, raw_path, bronze_path)

bronze_df.printSchema()


root
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- text_original: string (nullable = true)
 |-- text_normalized: string (nullable = true)
 |-- match_text: string (nullable = true)



## Prefilter And Extract Spec-0006 Candidates

The native prefilter is intentionally broad but cheap. It reduces the number of rows sent through the Python UDF.


In [5]:
prefiltered_bronze_df = prefilter_como_article_ground_vehicle_bronze_dataframe(bronze_df).persist()
prefiltered_count = prefiltered_bronze_df.count()
print(f"Prefiltered bronze rows: {prefiltered_count:,}")

candidates_df = prepare_como_article_ground_vehicle_dataframe(prefiltered_bronze_df).persist()
candidate_count = candidates_df.count()
print(f"Spec-0006 candidates: {candidate_count:,}")
candidates_df


Prefiltered bronze rows: 333


/usr/local/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Spec-0006 candidates: 204


DataFrame[candidate_id: string, source_file: string, original_line_id: bigint, segment_id: int, pattern_type: string, connector: string, connector_text: string, matched_text: string, candidate_full_text: string, text_before: string, tenor_text: string, tenor_lemma: string, tenor_confidence: string, ground_text: string, ground_lemma: string, ground_type: string, ground_source: string, vehicle_text: string, vehicle_lemma: string, vehicle_head: string, vehicle_head_lemma: string, vehicle_phrase_length_tokens: int, filter_label: string, reject_reason: string, confidence: double, needs_review: boolean, char_start: int, char_end: int, connector_start: int, connector_end: int, vehicle_start: int, vehicle_end: int]

In [6]:
candidates_df.printSchema()


root
 |-- candidate_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- pattern_type: string (nullable = true)
 |-- connector: string (nullable = true)
 |-- connector_text: string (nullable = true)
 |-- matched_text: string (nullable = true)
 |-- candidate_full_text: string (nullable = true)
 |-- text_before: string (nullable = true)
 |-- tenor_text: string (nullable = true)
 |-- tenor_lemma: string (nullable = true)
 |-- tenor_confidence: string (nullable = true)
 |-- ground_text: string (nullable = true)
 |-- ground_lemma: string (nullable = true)
 |-- ground_type: string (nullable = true)
 |-- ground_source: string (nullable = true)
 |-- vehicle_text: string (nullable = true)
 |-- vehicle_lemma: string (nullable = true)
 |-- vehicle_head: string (nullable = true)
 |-- vehicle_head_lemma: string (nullable = true)
 |-- vehicle_phrase_length_tokens: integer (nullable = true

## Write Gold Dataset And Visualization CSVs


In [7]:
write_como_article_ground_vehicle_parquet(candidates_df, REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH)
write_como_article_csv_outputs(
    candidates_df,
    ground_vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    vehicle_ground_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    ground_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    examples_path=REPO_ROOT / COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    review_sample_path=REPO_ROOT / COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
)


## Inspect Visualization Tables

These displays use small materialized pandas tables for readability. The durable CSV outputs above remain the source of truth for downstream visualization work.


In [9]:
from pyspark.sql.functions import col

def display_table(spark_df, limit=30):
    display(spark_df.limit(limit).toPandas())



### Evaluation Strategy Outputs


In [10]:

print(f"Evaluation dataset: {EVALUATION_DATASET_PATH}")
print(f"Evaluation outputs: {EVALUATION_OUTPUT_ROOT}")
display_table(
    evaluation_df.groupBy("pattern_type").count().orderBy("pattern_type"),
    limit=20,
)


Evaluation dataset: data/gold/evaluation/como_article_strategy_candidates
Evaluation outputs: outputs/evaluation


,pattern_type,count
0,como_article_ground_vehicle,204


### Ground -> Vehicle Pairs


In [11]:
display_table(
    como_article_ground_vehicle_counts_dataframe(candidates_df).select(
        col("ground_lemma").alias("ground"),
        col("vehicle_head_lemma").alias("vehicle_head"),
        "count",
    ),
    limit=40,
)


,ground,vehicle_head,count
0,cair,luva,49
1,cair,bomba,24
2,forte,touro,4
3,assentar,luva,3
4,caber,luva,3
5,encaixar,luva,3
6,flutuar,borboleta,3
7,rápido,flecha,3
8,rápido,raio,3
9,cair,verdadeiro,2


### Vehicle -> Ground Pairs


In [12]:
display_table(
    como_article_vehicle_ground_counts_dataframe(candidates_df).select(
        col("vehicle_head_lemma").alias("vehicle_head"),
        col("ground_lemma").alias("ground"),
        "count",
    ),
    limit=40,
)


,vehicle_head,ground,count
0,luva,cair,49
1,bomba,cair,24
2,touro,forte,4
3,borboleta,flutuar,3
4,flecha,rápido,3
5,luva,assentar,3
6,luva,caber,3
7,luva,encaixar,3
8,raio,rápido,3
9,avião,voar,2


### Ground Counts


In [13]:
display_table(como_article_ground_counts_dataframe(candidates_df), limit=40)


,ground_lemma,count
0,cair,83
1,trabalhar,15
2,forte,12
3,rápido,9
4,correr,6
5,duro,6
6,grande,6
7,voar,6
8,alto,5
9,encaixar,5


### Vehicle Counts


In [14]:
display_table(como_article_vehicle_counts_dataframe(candidates_df), limit=40)


,pattern_type,vehicle_head_clean,count
0,como_article_ground_vehicle,luva,58
1,como_article_ground_vehicle,bomba,25
2,como_article_ground_vehicle,raio,4
3,como_article_ground_vehicle,touro,4
4,como_article_ground_vehicle,verdadeiro,4
5,como_article_ground_vehicle,bala,3
6,como_article_ground_vehicle,borboleta,3
7,como_article_ground_vehicle,flecha,3
8,como_article_ground_vehicle,avião,2
9,como_article_ground_vehicle,caixa,2


### Examples


In [15]:
display_table(
    como_article_examples_dataframe(candidates_df, limit=100).select(
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "candidate_full_text",
    ),
    limit=50,
)


,pattern_type,ground_text,connector_text,vehicle_text_clean,candidate_full_text
0,como_article_ground_vehicle,alegre,como um,passarinho,"Ela abriu os olhos e saltou da cama, alegre co..."
1,como_article_ground_vehicle,alto,como uma,caixa,Para iniciar com tranqüilidade vamos sentar so...
2,como_article_ground_vehicle,alta,como uma,estrela,"gro vê-la, cego ser e não vê-la, eu preferira:..."
3,como_article_ground_vehicle,alta,como um,pau de sebo,"A criada, alta como um pau de sebo"
4,como_article_ground_vehicle,alto,como um,pássaro com asas de anjo,"Voarei além do céu, além das estrelas Voarei a..."
5,como_article_ground_vehicle,alto,como um,super heroi,Se você já sonhou que estava voando alto como ...
6,como_article_ground_vehicle,assenta,como uma,luva,"mundo tem lutado nasce da ideia de que ""o bem""..."
7,como_article_ground_vehicle,assenta,como uma,luva,Álvaro Magalhães é seguramente um daqueles a q...
8,como_article_ground_vehicle,assenta,como uma,luva no dia de hoje,"NA LISTA de livros que se pode ver [aqui], há ..."
9,como_article_ground_vehicle,baixo,como um,barco,empo eu supri minha sala com uma moderna répli...


### Review Sample


In [16]:
display_table(
    como_article_review_sample_dataframe(candidates_df, limit=100).select(
        "ground_type",
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "confidence",
        "needs_review",
        "candidate_full_text",
    ),
    limit=50,
)


,pattern_type,ground_text,connector_text,vehicle_text_clean,quality_reason,needs_review,candidate_full_text
0,como_article_ground_vehicle,Flutuar,como uma,borboleta,spec_0006_strict_filter,True,Flutuar como uma borboleta
1,como_article_ground_vehicle,Forte,como um,touro e alfabetizada,spec_0006_strict_filter,True,Forte como um touro e alfabetizada
2,como_article_ground_vehicle,Livre,como um,táxi,spec_0006_strict_filter,True,Livre como um táxi
3,como_article_ground_vehicle,Rápida,como uma,flecha,spec_0006_strict_filter,True,Rápida como uma flecha
4,como_article_ground_vehicle,Calmo,Como Uma,Bomba,spec_0006_strict_filter,True,Calmo Como Uma Bomba
5,como_article_ground_vehicle,Correu,como uma,bala para a pensão,spec_0006_strict_filter,True,Correu como uma bala para a pensão
6,como_article_ground_vehicle,Cego,como um,morcego,spec_0006_strict_filter,True,Cego como um morcego
7,como_article_ground_vehicle,Brilhante,como uma,caixa de celofane âmbar,spec_0006_strict_filter,True,Brilhante como uma caixa de celofane âmbar
8,como_article_ground_vehicle,voar,como um,avião,spec_0006_strict_filter,False,"a é bem sucedida, o módulo encontra a parte ma..."
9,como_article_ground_vehicle,caiu,como uma,bomba,spec_0006_strict_filter,False,Essa noticia caiu como uma bomba
